# INUA Sprint 3: Machine Learning Model Evaluation & Interpretation

**Student:** Michelle  
**Project:** INUA - Intelligent Mentorship Platform  
**Institution:** Strathmore University (Final Year Project - BICS Year 4)  
**Deliverables:** Dissertation Chapter 4 Performance Table 4.4 and Figures 4.8 - 4.10  

---

## Evaluation Objectives
1. **Held-Out Test Set Performance**: Evaluate on unseen $N=400$ test sessions (20% holdout).
2. **Comparative Classification Metrics**: Tabulate Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
3. **Confusion Matrix Analysis**: Inspect False Positives and False Negatives across models.
4. **Discrimination Capability**: Compare Receiver Operating Characteristic (ROC) trajectories.
5. **Feature Attribution & Explainability**: Identify top predictive features using Gini and gain metrics.

In [ ]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Publication Styling Configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.labelweight': 'semibold',
    'figure.titlesize': 14,
})

sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))
from preprocessing import run_preprocessing
from train_models import (
    evaluate_models,
    plot_confusion_matrices,
    plot_roc_curves,
    plot_feature_importances,
    initialize_models,
    perform_cross_validation
)

MODELS_DIR = os.path.join('..', 'saved_models')
FIG_DIR = os.path.join('..', 'figures')

X_train, X_test, y_train, y_test, feature_names = run_preprocessing(models_dir=MODELS_DIR)
print(f"Test set evaluated on: {len(X_test)} samples")

## 1. Table 4.4: Model Comparison Summary Metrics

In [ ]:
models = initialize_models(random_state=42)
cv_results = perform_cross_validation(models, X_train, y_train, n_splits=5)
trained_models, metrics_df, test_preds, test_probs = evaluate_models(
    models, cv_results, X_train, y_train, X_test, y_test
)

metrics_df

## 2. Figure 4.8: Confusion Matrices (Random Forest, AdaBoost, XGBoost)

In [ ]:
plot_confusion_matrices(y_test, test_preds, FIG_DIR)

# Display inline
from IPython.display import Image
Image(filename=os.path.join(FIG_DIR, 'figure_4_8_confusion_matrices.png'))

## 3. Figure 4.9: Receiver Operating Characteristic (ROC) Curves

In [ ]:
plot_roc_curves(y_test, test_probs, FIG_DIR)
Image(filename=os.path.join(FIG_DIR, 'figure_4_9_roc_curves.png'))

## 4. Figure 4.10: Top Predictive Feature Importances

In [ ]:
plot_feature_importances(trained_models, feature_names, FIG_DIR, top_k=10)
Image(filename=os.path.join(FIG_DIR, 'figure_4_10_feature_importances.png'))

## 5. Live Production Inference Demonstration

In [ ]:
from predict import MentorshipEffectivenessPredictor

predictor = MentorshipEffectivenessPredictor(models_dir=MODELS_DIR)

engaged_session = {
    'career_stage': 'Year4',
    'field_of_study': 'Computer Science',
    'career_interest': 'Software Engineering',
    'prior_sessions': 3,
    'mentor_experience_years': 8,
    'mentor_industry': 'Technology',
    'mentor_specialization': 'Backend Engineering',
    'mentor_availability': 'High',
    'session_type': 'Interview Preparation',
    'session_duration_minutes': 65,
    'attendance_status': 'Attended',
    'meeting_platform': 'Teams',
    'goal_category': 'Internship',
    'career_support_score': 4,
    'psychosocial_support_score': 4,
    'communication_quality': 5,
    'goal_progress_score': 4,
    'confidence_growth_score': 4,
    'mentor_responsiveness_score': 5,
}

result = predictor.predict_session(engaged_session, model_name='XGBoost')
print("Prediction Result:", result)